<a href="https://colab.research.google.com/github/ronan-cunha/machine-learning-course/blob/main/Notebooks%5CAula_5_construcao_de_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Estrutura do pipeline

  0. Input inicial: Dados brutos
  1. ETAPA 1: Imputação de Dados Ausentes (tratar NaNs)
  2. ETAPA 2: Detecção e Remoção de Outliers via IQR
  3. ETAPA 3: Padronização dos Dados
  4. ETAPA 4: Visualização dos dados (Boxplot)
  5. Output: Dataframe com dados limpos (sem outlier e NaNs) e padronizados

In [1]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Importar base de dados
iris = sns.load_dataset('iris')
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
# ----------------------------------------------------------
# Criação do Dataset Corrompido
# ----------------------------------------------------------
np.random.seed(42)
df_desafio = sns.load_dataset('iris')[
    ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
].copy()

# Inserindo NaNs e Outliers
mask_nan = np.random.rand(*df_desafio.shape) < 0.2
df_desafio[mask_nan] = np.nan
df_desafio.iloc[5, 0] = 9.0  # Outlier na sepal_length
df_desafio.iloc[1, 1] = 8.0  # Outlier na sepal_width

print('*** DATASET INICIAL ***')
df_desafio.head()

*** DATASET INICIAL ***


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,NaN,8.0,NaN,0.2
2,4.7,3.2,NaN,0.2
3,4.6,3.1,NaN,NaN
4,5.0,3.6,1.4,0.2


In [ ]:
# ----------------------------------------------------------
# ETAPA 1: Imputação de Dados Ausentes com IterativeImputer
# ----------------------------------------------------------
imp = IterativeImputer(max_iter=10, random_state=0)
df_imputed_array = imp.fit_transform(df_desafio)

# Convertendo de volta para DataFrame
cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df_imputed = pd.DataFrame(df_imputed_array, columns=cols)

# ----------------------------------------------------------
# ETAPA 2: Remoção de Outliers via IQR
# ----------------------------------------------------------
# Função vista na aula
def outliers_detector_iqr(df, column):
  Q1 = df[column].quantile(0.25)
  Q3 = df[column].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
  return outliers, lower_bound, upper_bound


# Filtrando outliers em 'sepal_length' e 'sepal_width'
df_clean = df_imputed.copy()

for col in df_imputed.columns:
  outliers, lower_b, upper_b = outliers_detector_iqr(df_clean, col)
  # Mantendo apenas as linhas dentro dos limites
  df_clean = df_clean[
      (df_clean[col] >= lower_b) & (df_clean[col] <= upper_b)
  ]

print('\n=== DATASET SEM OUTLIERS  ===')
print(df_clean.describe())


# ----------------------------------------------------------
# ETAPA 3: Padronização dos Dados com StandardScaler
# ----------------------------------------------------------
scaler = StandardScaler()
df_scaled_array = scaler.fit_transform(df_clean)

# DataFrame final totalmente tratado e padronizado
df_final = pd.DataFrame(df_scaled_array, columns=cols)

# ----------------------------------------------------------
# RESULTADO FINAL
# ----------------------------------------------------------
print('\n=== DATASET FINAL (TRATADO E PADRONIZADO) ===')
print(df_final.describe())

# Visualização em Boxplot do resultado
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_final)
plt.title('Dados Limpos, Sem Outliers Extremos e Padronizados (Média ~0, Std ~1)')
plt.show()

## Estrutura de uma classe

In [ ]:
class DataProcessor:
    def __init__(self):
        pass

In [ ]:
pipe0 = DataProcessor()
pipe0

In [ ]:
class DataProcessor:
    def __init__(self, imputation_strategy='media'):
        self.estrategia = imputation_strategy

In [ ]:
pipe1 = DataProcessor()
pipe1

In [ ]:
pipe1.estrategia

'media'

In [ ]:
class DataProcessor:
    def __init__(self, imputation_strategy='media'):
        self.estrategia = imputation_strategy

    def imputation_nan(self, vec):
        resultado = vec.where(~vec.isna(), vec.mean())
        return resultado

In [ ]:
x = pd.Series([np.nan, 2, 3, 4, 5])
x

,0
0,NaN
1,2.0
2,3.0
3,4.0
4,5.0


In [ ]:
pipe2 = DataProcessor()
pipe2.imputation_nan(x)

,0
0,3.5
1,2.0
2,3.0
3,4.0
4,5.0


In [ ]:
class DataProcessor:
    def __init__(self, imputation_strategy='media'):
        self.estrategia = imputation_strategy

    def _imputation_number(self, vec): # função interna (convenção inicar com underline)
        if self.estrategia == 'media':
            return vec.mean()
        elif self.estrategia == 'zero':
            return 0
        else:
            return 1

    def imputation_nan(self, vec):
        resultado = vec.where(~vec.isna(), self._imputation_number(vec))
        return resultado

In [ ]:
pipe3 = DataProcessor()
pipe3.imputation_nan(x)

,0
0,3.5
1,2.0
2,3.0
3,4.0
4,5.0


In [ ]:
pipe3 = DataProcessor(imputation_strategy='zero')
pipe3.imputation_nan(x)

,0
0,0.0
1,2.0
2,3.0
3,4.0
4,5.0


## Substituição de valores faltantes

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import SimpleImputer

class DataProcessor:
    def __init__(self, imputation_strategy='iterative'):
        self.estrategia = imputation_strategy

    def _get_imputer(self):
        if self.estrategia == 'iterative':
            return IterativeImputer(max_iter=10, random_state=0)
        elif self.estrategia == 'simple':
            return SimpleImputer(strategy='mean')
        else:
            raise ValueError("Estratégia de imputação inválida. Use 'iterative' ou 'simple'.")

    def imputation_nan(self, df):
        imputer = self._get_imputer()
        df_imputed_array = imputer.fit_transform(df)
        cols = df.columns
        return pd.DataFrame(df_imputed_array, columns=cols)

In [ ]:
pipe4 = DataProcessor()
df_imputed = pipe4.imputation_nan(df_desafio)
df_imputed.head()

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,sepal_length,sepal_width,petal_length,petal_width
0,5.100000,3.5,1.400000,0.200000
1,8.251882,8.0,0.306672,0.200000
2,4.700000,3.2,1.375364,0.200000
3,4.600000,3.1,1.631391,0.377929
4,5.000000,3.6,1.400000,0.200000


In [ ]:
pipe4 = DataProcessor(imputation_strategy='simple')
df_imputed = pipe4.imputation_nan(df_desafio)
df_imputed.head()

,sepal_length,sepal_width,petal_length,petal_width
0,5.100000,3.5,1.400000,0.200000
1,5.895652,8.0,3.679646,0.200000
2,4.700000,3.2,3.679646,0.200000
3,4.600000,3.1,3.679646,1.207627
4,5.000000,3.6,1.400000,0.200000


## Outlier detector



In [ ]:
class DataProcessor:
    def __init__(self, imputation_strategy='iterative', outlier_method='iqr'):
        self.estrategia = imputation_strategy
        self.outlier_method = outlier_method

    def _get_imputer(self):
        if self.estrategia == 'iterative':
            return IterativeImputer(max_iter=10, random_state=0)
        elif self.estrategia == 'simple':
            return SimpleImputer(strategy='mean')
        else:
            raise ValueError("Estratégia de imputação inválida. Use 'iterative' ou 'simple'.")

    def imputation_nan(self, df):
        imputer = self._get_imputer()
        df_imputed_array = imputer.fit_transform(df)
        cols = df.columns
        return pd.DataFrame(df_imputed_array, columns=cols)

    def outliers_detector_iqr(self, df, column):
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
        return outliers, lower_bound, upper_bound

    def remove_outliers(self, df_imputed):
        # Filtrando outliers em 'sepal_length' e 'sepal_width'
        df_clean = df_imputed.copy()
        for col in df_imputed.columns:
          outliers, lower_b, upper_b = self.outliers_detector_iqr(df_clean, col)
          # Mantendo apenas as linhas dentro dos limites
          df_clean = df_clean[
              (df_clean[col] >= lower_b) & (df_clean[col] <= upper_b)
              ]
        return df_clean

In [ ]:
pipe5 = DataProcessor()
df_imputed = pipe5.imputation_nan(df_desafio)
df_clean = pipe5.remove_outliers(df_imputed)
df_clean.head()

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.400000,0.200000
2,4.7,3.2,1.375364,0.200000
3,4.6,3.1,1.631391,0.377929
4,5.0,3.6,1.400000,0.200000
6,4.6,3.4,1.294775,0.300000


In [ ]:
def process_pipeline(self, df_original):
  df_imputed = self.imputation_nan(df_original.copy())
  df_clean = self.remove_outliers(df_imputed)
  return df_clean

In [ ]:
DataProcessor.process_pipeline = process_pipeline

In [ ]:
pipe6 = DataProcessor()
df_clean = pipe6.process_pipeline(df_desafio)
df_clean.head()

/usr/local/lib/python3.13/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.400000,0.200000
2,4.7,3.2,1.375364,0.200000
3,4.6,3.1,1.631391,0.377929
4,5.0,3.6,1.400000,0.200000
6,4.6,3.4,1.294775,0.300000


# Atividade
1. Adicione a etapa de padronização das variáveis na classe
2. Adicione a etapa de visualização gráfica: adicione o boxplot